In [15]:
import requests
import pandas as pd
import time
from datetime import datetime

Загрузка свечей с MOEX
интервал: 24 - дневные свечи

In [16]:
def load_moex_candles(ticker, start="2016-01-01", end="2025-12-31", interval=24):
    """
    Загрузка свечей с MOEX ISS API
    interval: 24 = дневные свечи
    """
    all_candles = []
    start_index = 0
    
    while True:
        url = (
            f"https://iss.moex.com/iss/engines/stock/markets/shares/"
            f"boards/TQBR/securities/{ticker}/candles.json"
        )
        params = {
            "from": start,
            "till": end,
            "interval": interval,
            "start": start_index
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        
        candles = data["candles"]["data"]
        columns = data["candles"]["columns"]
        
        if not candles:
            break
            
        all_candles.extend(candles)
        start_index += len(candles)
        
        if len(candles) < 500:
            break
            
        time.sleep(0.5)
    
    df = pd.DataFrame(all_candles, columns=columns)
    df["ticker"] = ticker
    return df

Загрузка бумаг, для анализа и обучения возьмем голубые фишки российского рынка

In [17]:
tickers = [
    "SBER", "GAZP", "LKOH", "VTBR", "GMKN",
    "NVTK", "TATN", "MGNT", "ROSN", "MTSS",
    "ALRS", "PHOR", "CHMF", "NLMK", "PIKK"
]

all_data = []

for ticker in tickers:
    try:
        df = load_moex_candles(ticker)
        all_data.append(df)
        print(f"{ticker}: {len(df)} свечей")
    except Exception as e:
        print(f"{ticker}: ошибка — {e}")
    time.sleep(1)

df_all = pd.concat(all_data, ignore_index=True)
print(f"\nВсего записей: {len(df_all)}")

SBER: 2577 свечей
GAZP: 2577 свечей
LKOH: 2577 свечей
VTBR: 2573 свечей
GMKN: 2573 свечей
NVTK: 2573 свечей
TATN: 2575 свечей
MGNT: 2577 свечей
ROSN: 2577 свечей
MTSS: 2576 свечей
ALRS: 2577 свечей
PHOR: 2575 свечей
CHMF: 2577 свечей
NLMK: 2577 свечей
PIKK: 2577 свечей

Всего записей: 38638


Приведем данные к нормальному виду

In [18]:
df_all = df_all.rename(columns={
    "open": "Open",
    "close": "Close",
    "high": "High",
    "low": "Low",
    "volume": "Volume",
    "begin": "Date"
})

df_all = df_all[["Date", "Open", "High", "Low", "Close", "Volume", "ticker"]]

df_all["Date"] = pd.to_datetime(df_all["Date"])

df_all = df_all.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(df_all.head(10))
print(f"\nФорма данных: {df_all.shape}")
print(f"\nТикеры: {df_all['ticker'].unique()}")

        Date   Open   High    Low  Close    Volume ticker
0 2016-01-04  56.00  56.78  55.20  55.21   1547200   ALRS
1 2016-01-05  55.01  57.80  54.66  57.80   5005100   ALRS
2 2016-01-06  57.50  58.90  56.86  57.69   5420100   ALRS
3 2016-01-11  56.90  57.28  55.63  56.11   6835600   ALRS
4 2016-01-12  56.06  58.40  55.50  58.34   6497700   ALRS
5 2016-01-13  58.27  59.18  57.17  57.61   7263900   ALRS
6 2016-01-14  57.45  58.46  56.15  57.26   5659600   ALRS
7 2016-01-15  56.82  58.04  53.62  53.90  12329600   ALRS
8 2016-01-18  53.05  54.09  52.33  53.31   7691800   ALRS
9 2016-01-19  53.90  55.45  53.45  54.09   8830100   ALRS

Форма данных: (38638, 7)

Тикеры: ['ALRS' 'CHMF' 'GAZP' 'GMKN' 'LKOH' 'MGNT' 'MTSS' 'NLMK' 'NVTK' 'PHOR'
 'PIKK' 'ROSN' 'SBER' 'TATN' 'VTBR']


In [19]:
df_all.to_csv("../data/raw_candles.csv", index=False)
print("Сохранено в data/raw_candles.csv")

Сохранено в data/raw_candles.csv


In [20]:
df = pd.read_csv("../data/raw_candles.csv")

# Общая инфа
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.isnull().sum())
print(df.groupby("ticker")["Date"].count().sort_values())


(38638, 7)
Date       object
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
ticker     object
dtype: object
         Date   Open   High    Low  Close   Volume ticker
0  2016-01-04  56.00  56.78  55.20  55.21  1547200   ALRS
1  2016-01-05  55.01  57.80  54.66  57.80  5005100   ALRS
2  2016-01-06  57.50  58.90  56.86  57.69  5420100   ALRS
3  2016-01-11  56.90  57.28  55.63  56.11  6835600   ALRS
4  2016-01-12  56.06  58.40  55.50  58.34  6497700   ALRS
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
ticker    0
dtype: int64
ticker
GMKN    2573
NVTK    2573
VTBR    2573
PHOR    2575
TATN    2575
MTSS    2576
ALRS    2577
CHMF    2577
GAZP    2577
LKOH    2577
MGNT    2577
NLMK    2577
PIKK    2577
ROSN    2577
SBER    2577
Name: Date, dtype: int64


In [21]:
df["Date"] = pd.to_datetime(df["Date"])
print(df.dtypes)
df.to_csv("../data/raw_candles.csv", index=False)


Date      datetime64[ns]
Open             float64
High             float64
Low              float64
Close            float64
Volume             int64
ticker            object
dtype: object
